# Lesson 5: AI Applications

**Module 1:** Introduction to Artificial Intelligence

**Lesson Objectives:**
- Describe major AI application areas
- Map applications to AI paradigms
- Identify real-world use cases

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries loaded")

## Simple Recommendation System

Let us build a basic recommendation system using collaborative filtering.

In [ ]:
# User-item interaction matrix
# Rows: users, Columns: items (movies), Values: ratings (1-5, 0 = not rated)
ratings = np.array([
    [5, 4, 0, 0, 1],  # User 0
    [4, 5, 0, 0, 2],  # User 1
    [0, 0, 5, 4, 0],  # User 2
    [0, 0, 4, 5, 3],  # User 3
    [1, 2, 0, 3, 5],  # User 4
])

movies = ['Movie_A', 'Movie_B', 'Movie_C', 'Movie_D', 'Movie_E']
users = ['User_0', 'User_1', 'User_2', 'User_3', 'User_4']

print("User-Item Rating Matrix:")
print(pd.DataFrame(ratings, index=users, columns=movies))

In [ ]:
def collaborative_filtering(ratings, user_id, n_recommendations=2):
    """
    Simple collaborative filtering recommendation.
    Finds similar users and recommends items they liked.
    """
    n_users, n_items = ratings.shape
    
    # Get the target user's ratings
    target = ratings[user_id]
    
    # Compute similarity with all other users (cosine similarity)
    similarities = []
    for i in range(n_users):
        if i == user_id:
            continue
        # Only consider items both users rated
        both_rated = (target > 0) & (ratings[i] > 0)
        if both_rated.sum() == 0:
            sim = 0
        else:
            # Simple similarity: mean rating difference on common items
            diff = np.abs(target[both_rated] - ratings[i][both_rated])
            sim = 1 / (1 + diff.mean())
        similarities.append((i, sim))
    
    # Find the most similar user
    similarities.sort(key=lambda x: x[1], reverse=True)
    most_similar = similarities[0][0]
    
    # Recommend items the similar user liked that target hasn't seen
    candidate_items = []
    for j in range(n_items):
        if target[j] == 0 and ratings[most_similar][j] >= 4:
            candidate_items.append((j, ratings[most_similar][j]))
    
    candidate_items.sort(key=lambda x: x[1], reverse=True)
    
    return movies[most_similar], [movies[j] for j, _ in candidate_items[:n_recommendations]]


# Test
for uid in range(5):
    similar_user, recs = collaborative_filtering(ratings, uid)
    print(f"User {uid}: Most similar to {similar_user}")
    print(f"  Recommended: {recs}\n")

## Simple Image Classification Concept

Let us demonstrate a simple image-like classification using pixel data.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Load digits dataset (8x8 images of handwritten digits)
digits = load_digits()
X_digits, y_digits = digits.data, digits.target

print(f"Dataset: {X_digits.shape[0]} images, {X_digits.shape[1]} features")
print(f"Image size: 8x8 = 64 pixels")
print(f"Classes: {digits.target_names}")

# Show a sample image
plt.figure(figsize=(8, 3))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(digits.images[i], cmap='gray')
    plt.title(f"Label: {digits.target[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Train a classifier (Computer Vision application)
from sklearn.model_selection import train_test_split

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_digits, y_digits, test_size=0.3, random_state=42
)

rf_digits = RandomForestClassifier(n_estimators=100, random_state=42)
rf_digits.fit(Xd_train, yd_train)
yd_pred = rf_digits.predict(Xd_test)

print("Classification Report (Digit Recognition):")
print(classification_report(yd_test, yd_pred, target_names=[str(i) for i in range(10)]))

## Simple NLP: Sentiment Analysis

Let us demonstrate a basic text classification (sentiment analysis).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Sample reviews
reviews = [
    "This product is amazing and works perfectly",
    "Terrible quality, broke after one use",
    "Good value for the price, very satisfied",
    "Waste of money, do not buy",
    "Works as expected, decent product",
    "Horrible customer service and faulty item",
]
labels = [1, 0, 1, 0, 1, 0]  # 1 = positive, 0 = negative

# Convert text to features (bag of words)
vectorizer = CountVectorizer()
X_reviews = vectorizer.fit_transform(reviews)

print("Feature names (vocabulary):")
print(vectorizer.get_feature_names_out()[:20])
print(f"\nFeature matrix shape: {X_reviews.shape}")

# Train a simple classifier
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_reviews, labels)

# Test on new reviews
new_reviews = [
    "Absolutely love this, fantastic product",
    "Disappointed, poor quality",
    "Pretty good, would recommend",
]
X_new = vectorizer.transform(new_reviews)
predictions = nb.predict(X_new)

print("\nSentiment Predictions:")
for review, pred in zip(new_reviews, predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"  '{review}' → {sentiment}")

## Application Mapping

Let us create a structured overview of AI applications.

In [ ]:
applications = {
    "Computer Vision": [
        "Image classification", "Object detection", "Face recognition",
        "Medical imaging", "Self-driving perception", "OCR"
    ],
    "NLP": [
        "Sentiment analysis", "Machine translation", "Summarization",
        "Named Entity Recognition", "Question answering", "Chatbots"
    ],
    "Robotics": [
        "Navigation", "Manipulation", "Path planning",
        "Human-robot interaction", "Autonomous vehicles"
    ],
    "Recommendation": [
        "Collaborative filtering", "Content-based filtering",
        "Hybrid recommenders", "Context-aware recommenders"
    ],
    "Generative AI": [
        "Text generation", "Image generation", "Code generation",
        "Music generation", "Video generation"
    ]
}

for area, tasks in applications.items():
    print(f"\n{area}:")
    for task in tasks:
        print(f"  • {task}")

## Exercises

1. Improve the collaborative filtering function to use a proper similarity metric (cosine similarity).
2. Train a classifier on a different dataset from sklearn (e.g., load_wine) and identify which application area it belongs to.
3. Research one AI application not mentioned in this lesson and write a markdown cell describing it.

In [ ]:
# Exercise 1: Improved collaborative filtering with cosine similarity
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    both_nonzero = (a > 0) & (b > 0)
    if both_nonzero.sum() == 0:
        return 0
    a_filt = a[both_nonzero]
    b_filt = b[both_nonzero]
    return np.dot(a_filt, b_filt) / (np.linalg.norm(a_filt) * np.linalg.norm(b_filt) + 1e-10)


def improved_cf(ratings, user_id, n_recommendations=2):
    """Improved collaborative filtering using cosine similarity."""
    n_users, n_items = ratings.shape
    target = ratings[user_id]
    
    similarities = [(i, cosine_similarity(target, ratings[i])) 
                    for i in range(n_users) if i != user_id]
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    most_similar = similarities[0][0]
    
    candidate_items = [(j, ratings[most_similar][j])
                      for j in range(n_items)
                      if target[j] == 0 and ratings[most_similar][j] >= 4]
    candidate_items.sort(key=lambda x: x[1], reverse=True)
    
    return [movies[j] for j, _ in candidate_items[:n_recommendations]]


print("Improved Recommendations:")
for uid in range(5):
    recs = improved_cf(ratings, uid)
    print(f"  User {uid}: {recs}")

## Summary

- Five major AI application areas with different technologies
- Each area has specific tasks and success metrics
- Recommendation, CV, and NLP are the most deployed in industry
- Generative AI is the newest and fastest-evolving area
- Most real-world AI systems combine multiple application areas